<div dir="rtl" align="right">

# تقييمُ ICA عبرَ FFT

**مجموعةُ البياناتِ**: PhysioNet Auditory EEG  
**القنواتُ**: P4, Cz, F8, T7  
**معدّلُ أخذِ العيناتِ**: 200 Hz  
**المُشاركُ**: 1

---

## نظرةٌ عامّةٌ

نُقارنُ الطيفَ التردديَّ للقناةِ P4 قبلَ وبعدَ تنظيفِ ICA (استبعادُ المكوّنِ الأكثرِ تبايناً) لِتقييمِ تأثيرِ إزالةِ الآثارِ على نطاقاتِ موجاتِ الدماغ.

## المُخرجاتُ المُتوقّعةُ

- طيفٌ قبلَ التنظيفِ في الأعلى
- طيفٌ بعدَ التنظيفِ في الأسفل
- نطاقاتُ موجاتِ الدماغِ الخمسُ مُظلّلةٌ في كليهما

## المُعاملاتُ الأساسيةُ

| المُعاملُ | القيمةُ | المعنى |
| --- | --- | --- |
| القناةُ | P4 | المنطقةُ الجداريةُ |
| n_components | 4 | عددُ مكوناتِ ICA |
| exclude | أعلى تباينٍ | المكوّنُ المُستبعدُ |

</div>

<div dir="rtl" align="right">

## 1. تثبيتُ المكتباتِ

</div>

In [ ]:
!pip install mne scikit-learn EMD-signal scipy numpy plotly wfdb


<div dir="rtl" align="right">

## 2. استنساخُ المستودعِ وتنزيلُ بياناتِ مُشاركٍ واحدٍ

نَنزّلُ مُشاركًا واحدًا فقط (`--subjects 1`) لتسريعِ التجربةِ في بيئةِ Colab.

</div>

In [ ]:
import os
if not os.path.exists('python-EEG-Arabic-Resources'):
    !git clone https://github.com/NibrasAz7/python-EEG-Arabic-Resources.git
os.chdir('python-EEG-Arabic-Resources')


In [ ]:
from pathlib import Path
data_dir = Path('data/local')
if not data_dir.exists() or not any(data_dir.glob('*.dat')):
    !python data/download_local.py --output data/local --subjects 1


<div dir="rtl" align="right">

## 3. تحميلُ إشارةِ EEG

نحمّلُ تسجيلَ المُشاركِ 1 في التجربةِ 1، الجلسةِ 2.

</div>

In [ ]:
import numpy as np
from utils.eeg_loader import load_local_eeg

timestamps, eeg_data, ch_names = load_local_eeg(
    data_dir='data/local', subject=1, experiment=1, session=2
)
fs = 200

print(f'Channels: {ch_names}')
print(f'Signal length: {len(eeg_data)} samples ({len(eeg_data)/fs:.1f} seconds)')


<div dir="rtl" align="right">

## 4. تطبيقُ ICA ومقارنةُ الطيفِ

نُطبّقُ ICA ونَستبعدُ المكوّنَ الأكثرَ تبايناً، ثمّ نُقارنُ طيفَ الإشارةِ قبلَ وبعدَ التنظيفِ.

</div>

In [ ]:
import mne
from scipy.fft import fft, fftfreq

info = mne.create_info(ch_names, sfreq=fs, ch_types='eeg')
raw = mne.io.RawArray(eeg_data.T * 1e-6, info, verbose=False)

# NOTE: ICA works best with more channels than components.
ica = mne.preprocessing.ICA(
    n_components=3, random_state=97, max_iter=800, verbose=False
)
ica.fit(raw, verbose=False)

component_variances = np.var(ica.get_sources(raw).get_data(), axis=1)
exclude_idx = int(np.argmax(component_variances))
ica.exclude = [exclude_idx]
cleaned_raw = ica.apply(raw.copy(), verbose=False)
cleaned_data = cleaned_raw.get_data()[0] * 1e6
original_data = eeg_data[:, 0]

def compute_spectrum(data, fs):
    spectrum = fft(data)
    freqs = fftfreq(len(data), 1 / fs)
    magnitude = np.abs(spectrum)
    pos_mask = freqs >= 0
    return freqs[pos_mask], magnitude[pos_mask]

freqs_orig, mag_orig = compute_spectrum(original_data, fs)
freqs_clean, mag_clean = compute_spectrum(cleaned_data, fs)
print(f'Excluded component: IC{exclude_idx}')


<div dir="rtl" align="right">

## 5. رسمٌ تفاعليٌّ

**علامَ تُلاحظُ؟**

- الفرقُ بينَ الطيفينِ يَدلُّ على تأثيرِ التنظيفِ
- النطاقاتُ المُظلّلةُ تُمثّلُ موجاتِ الدماغِ الخمسَ
- استخدمْ التكبيرَ لِفحصِ نطاقاتٍ تردديّةٍ مُحدّدةٍ


</div>

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

BANDS = [
    ('Delta', 0.5, 4, 'green'),
    ('Theta', 4, 8, 'blue'),
    ('Alpha', 8, 13, 'orange'),
    ('Beta', 13, 30, 'red'),
    ('Gamma', 30, 80, 'purple'),
]

fig = make_subplots(rows=2, cols=1, shared_xaxes=False,
                    subplot_titles=('FFT Before ICA - Channel P4',
                                    'FFT After ICA - Channel P4'))
fig.add_trace(go.Scatter(x=freqs_orig, y=mag_orig, name='Before',
                         line=dict(color='black', width=0.8)), row=1, col=1)
fig.add_trace(go.Scatter(x=freqs_clean, y=mag_clean, name='After',
                         line=dict(color='green', width=0.8)), row=2, col=1)
for name, fmin, fmax, color in BANDS:
    fig.add_vrect(x0=fmin, x1=fmax, fillcolor=color, opacity=0.1,
                  line_width=0, row=1, col=1)
    fig.add_vrect(x0=fmin, x1=fmax, fillcolor=color, opacity=0.1,
                  line_width=0, row=2, col=1)
fig.update_xaxes(range=[0, 80], row=1, col=1)
fig.update_xaxes(range=[0, 80], row=2, col=1)
fig.update_layout(height=700, title_text='FFT Evaluation - Before vs After ICA',
                  xaxis_title='Frequency (Hz)', xaxis2_title='Frequency (Hz)',
                  yaxis_title='Magnitude', yaxis2_title='Magnitude',
                  showlegend=False)
fig.show()


<div dir="rtl" align="right">

## خلاصةٌ

- مقارنةُ الطيفِ تَكشفُ تأثيرَ التنظيفِ على النطاقاتِ التردديّةِ
- استبعادُ المكوّنِ الأكثرِ تبايناً يُزيلُ الآثارَ ذاتَ السعةِ العالية
- النطاقاتُ الدماغيّةُ الأساسيّةُ تَبقى بعدَ التنظيفِ
- التقييمُ البصريُّ يُكمّلُ التقييمَ الكمّيَّ (SNR)


</div>